# Deep Learning Text Generation Learning Project
## Text Generation using **Vanilla RNN, LSTM, and GRU**

This notebook is built for **students and beginners** to understand how sequence models learn:
- grammar
- sentence flow
- contextual dependencies
- next-word prediction
- text generation

Goal: Compare **Simple RNN vs LSTM vs GRU** on the same text corpus and understand why gated architectures perform better.

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense

# Basic text corpus
corpus = '''
Artificial intelligence is a rapidly expanding field.
Machine learning is a subset of AI that enables systems to learn from data.
Neural networks are inspired by the human brain and are fundamental to deep learning.
Natural language processing allows computers to understand and generate human language.
Computer vision focuses on enabling machines to see and interpret images.
'''

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus])
total_words = len(tokenizer.word_index) + 1

# Sequence creation
input_sequences = []
for line in corpus.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

In [ ]:
# 1. Vanilla RNN
rnn_model = Sequential([
    Embedding(total_words, 16),
    SimpleRNN(32),
    Dense(total_words, activation='softmax')
])

rnn_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
rnn_model.fit(X, y, epochs=10, verbose=0)

In [ ]:
# 2. LSTM
lstm_model = Sequential([
    Embedding(total_words, 16),
    LSTM(32),
    Dense(total_words, activation='softmax')
])

lstm_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_model.fit(X, y, epochs=10, verbose=0)

In [ ]:
# 3. GRU
gru_model = Sequential([
    Embedding(total_words, 16),
    GRU(32),
    Dense(total_words, activation='softmax')
])

gru_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
gru_model.fit(X, y, epochs=10, verbose=0)

In [ ]:
# Generate text function
def generate_text(model, seed_text, next_words=5):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]
        
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

print("RNN Output:", generate_text(rnn_model, "deep learning", 5))
print("LSTM Output:", generate_text(lstm_model, "deep learning", 5))
print("GRU Output:", generate_text(gru_model, "deep learning", 5))

### Conclusion
* All three models (RNN, LSTM, and GRU) were built and trained on the text corpus to predict the next word.
* The generated text quality is somewhat random because the dataset is very small and the epochs were kept low.
* Generally, LSTM and GRU are designed to handle memory better than a Vanilla RNN, which helps them remember context over longer sentences.